# 0. Configuration and reproducibility

This notebook is the entry point for the interactive documentation. A YAML configuration remains the single source of truth: the notebook passes it to `simcast.config.load_config`, which is exactly what every CLI command does. Notebook-only choices are limited to the same arguments exposed by the CLI, such as a cache directory and output directory.

A full-group configuration fixes the ordered physical entity group. Do not use a notebook to select a prefix or random subset: that would no longer be the evaluated protocol.

In [ ]:
import os
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd() / 'notebooks' if (Path.cwd() / 'notebooks').is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from _helpers import REPOSITORY_ROOT, cache_path, resolve_config
os.chdir(REPOSITORY_ROOT)

CONFIG_FILE = 'configs/liander2024_transformer.yaml'
OVERRIDES: tuple[str, ...] = ()
# Example: OVERRIDES = ('chronos.device=cpu', 'training.epochs=20')
config = resolve_config(CONFIG_FILE, OVERRIDES)
print(REPOSITORY_ROOT)
print(config.model_dump_json(indent=2))

## What is fixed, and what may be changed

The configuration defines the entity type, frozen Chronos revision, forecast geometry, data availability rules, PIT repair, features, dependence model, training hyperparameters, sampling, and evaluation scores. `OVERRIDES` is appropriate for an intentional experimental change, but must be recorded: the CLI and notebook both save a resolved configuration with output artifacts.

`protocol.full_group_only: true` has three machine-enforced consequences: subset training is disabled; variable-cardinality evaluation is disabled; and the full ordered entity list is stored in run metadata. The group is a physical experimental unit, not a model input that may shrink when data are missing.

In [ ]:
group_config = config.protocol
print('protocol:', group_config.name)
print('full group only:', group_config.full_group_only)
print('cache:', cache_path(config))
print('entity type:', config.data.entity_type)
assert group_config.full_group_only
assert not config.subset_training.enabled
assert not config.evaluation.variable_k_sizes

## CLI equivalence

The following command and the notebook configuration resolve to the same `SimcastConfig`. The notebook will call the same underlying `*_from_config` functions rather than subprocesses, so exceptions, metadata, and numerical behavior are shared.

```bash
uv run python -m simcast.cli.run_experiment \
  --config configs/liander2024_transformer.yaml \
  --set chronos.device=cpu
```

Use a unique output directory for every actual experiment. Reusing a cache is normal; reusing an existing experiment-output directory is rejected to protect provenance.